# Étude de la pollution des stations de la RATP

Ce notebook analyse le jeu de données d'Île-de-France Mobilités sur la qualité de l'air dans les stations souterraines du réseau francilien, réalisé avec Airparif. On charge d'abord les données brutes puis on les nettoie avant de calculer un niveau de pollution moyen par ligne.

Le fichier est téléchargé directement depuis la source officielle au moment de l'exécution. Les noms exacts de colonnes sont détectés automatiquement dans le code (via des mots clés comme « pollution » ou « ligne ») car ils peuvent varier légèrement d'un export à l'autre, donc vérifie bien l'affichage des colonnes à la cellule suivante avant de poursuivre.

In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import matplotlib.pyplot as plt

URL = "https://www.data.gouv.fr/api/1/datasets/r/efb9ab99-4c52-4722-be5a-245c5322ab33"

try:
    df = pd.read_csv(URL, sep=";")
    if df.shape[1] == 1:
        raise ValueError("separateur incorrect")
except Exception:
    df = pd.read_csv(URL, sep=",")

print(df.shape)

On regarde les 10 premières lignes du jeu de données ainsi que la liste complète des colonnes disponibles.

In [ ]:
df.head(10)

In [ ]:
print(df.columns.tolist())

Pour notre étude on n'a pas besoin des coordonnées GPS ni des autres colonnes techniques situées au milieu du tableau. On garde uniquement les 4 premières colonnes et les 3 dernières, le reste est coupé.

In [ ]:
df = pd.concat([df.iloc[:, :4], df.iloc[:, -3:]], axis=1)
df.head(10)

On définit une petite fonction de normalisation (minuscules, sans accents) pour repérer les colonnes et les valeurs sans être gêné par la casse ou les accents. On repère ainsi la colonne du niveau de pollution et celle de la ligne de transport.

In [ ]:
def normalize(s):
    if pd.isna(s):
        return s
    s = str(s).strip().lower()
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    return s

def find_col(dataframe, keyword):
    for c in dataframe.columns:
        if keyword in normalize(c):
            return c
    return None

pol_col = find_col(df, "pollution")
ligne_col = find_col(df, "ligne")
print("colonne pollution :", pol_col)
print("colonne ligne :", ligne_col)

On affiche les différentes valeurs présentes dans la colonne du niveau de pollution, sans doublons, pour connaître les catégories utilisées et repérer les absences de mesure.

In [ ]:
print(df[pol_col].unique())
print("valeurs manquantes :", df[pol_col].isna().sum())

On remplace les absences de mesure par la catégorie moyenne, qui sert de valeur par défaut la plus neutre en l'absence d'information.

In [ ]:
df[pol_col] = df[pol_col].fillna("moyen")
print(df[pol_col].isna().sum())

On ajoute une colonne numérique « valeur pollution » à partir de la catégorie textuelle, avec la correspondance suivante : station aérienne = 0.5, faible = 1, moyen = 2, élevé = 3.

In [ ]:
mapping = {
    "station aerienne": 0.5,
    "faible": 1,
    "moyen": 2,
    "eleve": 3,
}

df["valeur pollution"] = df[pol_col].apply(lambda x: mapping.get(normalize(x), np.nan))

non_reconnues = df.loc[df["valeur pollution"].isna(), pol_col].unique()
if len(non_reconnues) > 0:
    print("valeurs non reconnues par le mapping :", non_reconnues)

df[[pol_col, "valeur pollution"]].head(10)

On calcule, pour chaque ligne de transport, la somme des niveaux de pollution divisée par le nombre de stations de cette ligne. Ce calcul inclut ici toutes les stations, y compris les stations aériennes.

In [ ]:
grouped_all = df.groupby(ligne_col)["valeur pollution"].agg(["sum", "count"])
grouped_all["pollution_moyenne"] = grouped_all["sum"] / grouped_all["count"]

top10_all = grouped_all.sort_values("pollution_moyenne", ascending=False).head(10)
bottom10_all = grouped_all.sort_values("pollution_moyenne", ascending=True).head(10)

print("10 lignes les plus polluées (toutes stations) :")
display(top10_all)
print("10 lignes les moins polluées (toutes stations) :")
display(bottom10_all)

On refait le même calcul en excluant cette fois les stations aériennes, aussi bien dans la somme des niveaux de pollution que dans le nombre de stations comptées. On vérifie d'abord qu'il reste au moins une station non aérienne par ligne pour éviter une division par zéro.

In [ ]:
mask_non_aerien = df[pol_col].apply(lambda x: normalize(x) != "station aerienne")
df_non_aerien = df[mask_non_aerien]

lignes_toutes = set(df[ligne_col].unique())
lignes_non_aerien = set(df_non_aerien[ligne_col].unique())
lignes_exclues = lignes_toutes - lignes_non_aerien

if len(lignes_exclues) > 0:
    print("lignes exclues car aucune station non aerienne :", lignes_exclues)
else:
    print("chaque ligne possede au moins une station non aerienne")

grouped_sol = df_non_aerien.groupby(ligne_col)["valeur pollution"].agg(["sum", "count"])
grouped_sol["pollution_moyenne"] = grouped_sol["sum"] / grouped_sol["count"]

top10_sol = grouped_sol.sort_values("pollution_moyenne", ascending=False).head(10)
bottom10_sol = grouped_sol.sort_values("pollution_moyenne", ascending=True).head(10)

print("10 lignes les plus polluees (stations souterraines uniquement) :")
display(top10_sol)
print("10 lignes les moins polluees (stations souterraines uniquement) :")
display(bottom10_sol)

On visualise enfin la relation entre le nombre de stations d'une ligne et son niveau de pollution moyen, un point représentant une ligne de transport. Premier nuage de points avec toutes les stations.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(grouped_all["count"], grouped_all["pollution_moyenne"])
plt.xlabel("Nombre de stations sur la ligne")
plt.ylabel("Pollution moyenne")
plt.title("Pollution moyenne par ligne en fonction du nombre de stations (toutes stations)")
plt.grid(True, alpha=0.3)
plt.show()

Second nuage de points, cette fois en ne comptabilisant que les stations souterraines.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(grouped_sol["count"], grouped_sol["pollution_moyenne"])
plt.xlabel("Nombre de stations souterraines sur la ligne")
plt.ylabel("Pollution moyenne")
plt.title("Pollution moyenne par ligne en fonction du nombre de stations (hors aerien)")
plt.grid(True, alpha=0.3)
plt.show()